In [9]:
# Import SparkSession class from pyspark.sql module
# SparkSession is the entry point to use Spark functionality in PySpark
# Without SparkSession, we cannot create DataFrames or run Spark jobs

from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .appName("Regression_Tutorial") \
    .getOrCreate()
# Create or get an existing Spark session
# SparkSession.builder is used to configure the Spark application

#spark = SparkSession.builder \

    # Set the name of the Spark application
    # This name appears in Spark UI and logs
    #.appName("BigData_Regression_Tutorial") \

    # getOrCreate() does two things:
    # 1️⃣ If a Spark session already exists → it returns that session
    # 2️⃣ If no session exists → it creates a new Spark session

   # .getOrCreate()

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
df = spark.read.csv("/content/drive/MyDrive/Big Data Foundation/housing_bigdata.csv",
                    header=True,          # First row contains column names
                    inferSchema=True)     # Automatically detect data types (int, double, etc.)

df.printSchema()   # Displays column names, data types, and nullability information

df.show()         # Shows first 5 rows of the DataFrame (triggers execution)


root
 |-- area: integer (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: integer (nullable = true)
 |-- floors: integer (nullable = true)
 |-- parking: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- location_score: double (nullable = true)
 |-- school_rating: double (nullable = true)
 |-- hospital_distance: double (nullable = true)
 |-- crime_rate: double (nullable = true)
 |-- population_density: integer (nullable = true)
 |-- median_income: integer (nullable = true)
 |-- highway_distance: double (nullable = true)
 |-- price: double (nullable = true)

+----+--------+---------+------+-------+---+--------------+-------------+-----------------+----------+------------------+-------------+----------------+----------+
|area|bedrooms|bathrooms|floors|parking|age|location_score|school_rating|hospital_distance|crime_rate|population_density|median_income|highway_distance|     price|
+----+--------+---------+------+-------+---+--------------+--------

In [11]:
df = df.dropna()

df.show(5)

+----+--------+---------+------+-------+---+--------------+-------------+-----------------+----------+------------------+-------------+----------------+----------+
|area|bedrooms|bathrooms|floors|parking|age|location_score|school_rating|hospital_distance|crime_rate|population_density|median_income|highway_distance|     price|
+----+--------+---------+------+-------+---+--------------+-------------+-----------------+----------+------------------+-------------+----------------+----------+
|1360|       3|        3|     1|      2| 24|          9.77|          5.2|             8.04|      6.14|             20714|        87837|            3.95| 758351.48|
|4272|       5|        2|     3|      0| 13|          6.93|         4.04|             3.63|      7.48|             19976|        65041|             7.5|1030963.01|
|3592|       2|        3|     3|      2|  0|          6.84|         6.42|             2.38|      1.91|             14830|       103377|            3.22|1076900.82|
| 966|       6| 

In [12]:
from pyspark.ml.feature import VectorAssembler   # Import VectorAssembler to combine feature columns into a single vector column

feature_columns = [                              # List of independent (input) feature column names
    'area','bedrooms','bathrooms','floors','parking',
    'age','location_score','school_rating',
    'hospital_distance','crime_rate',
    'population_density','median_income','highway_distance'
]

assembler = VectorAssembler(                     # Create VectorAssembler object
    inputCols=feature_columns,                   # Columns to combine into feature vector
    outputCol="features"                         # Name of the new output column containing combined features
)


In [13]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features"
)

In [14]:
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)   # Split dataset into 80% training data and 20% testing data (seed ensures same split every time)


In [16]:
from pyspark.ml.regression import LinearRegression   # Import Linear Regression algorithm from PySpark ML library

lr = LinearRegression(                               # Create Linear Regression model object
    featuresCol='scaled_features',                          # Column containing input scaled vectors (X)
    labelCol='price'                                 # Column containing target variable (Y)
)


In [18]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[assembler, scaler, lr])

model = pipeline.fit(train_data)

In [19]:
predictions = model.transform(test_data)

predictions.select("price", "prediction").show(5)

+---------+------------------+
|    price|        prediction|
+---------+------------------+
|595773.94| 612433.2448228666|
|624128.65| 570258.3521088869|
|495630.78| 489446.0988489079|
|636405.86| 594606.6373909729|
|420972.78|467580.28207348177|
+---------+------------------+
only showing top 5 rows


In [20]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_r2 = RegressionEvaluator(
    labelCol="price",
    predictionCol="prediction",
    metricName="r2"
)

evaluator_rmse = RegressionEvaluator(
    labelCol="price",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_mae = RegressionEvaluator(
    labelCol="price",
    predictionCol="prediction",
    metricName="mae"
)

print("R2:", evaluator_r2.evaluate(predictions))
print("RMSE:", evaluator_rmse.evaluate(predictions))
print("MAE:", evaluator_mae.evaluate(predictions))

R2: 0.9867677526899729
RMSE: 24984.032027076075
MAE: 19937.287985891442


In [21]:
lr_model = model.stages[-1]

print("Coefficients:", lr_model.coefficients)
print("Intercept:", lr_model.intercept)

Coefficients: [194806.23273373902,34172.376908378195,16802.992474575563,-3.8930266213021696,-8.673511419826788,-14426.570881297155,51934.89355355797,40406.46942093737,9.717298387794376,-25964.202369145973,-7.459100161466389,43294.31657710928,-17.924983111594134]
Intercept: 40.715352316591265


In [22]:
summary = lr_model.summary

print("Train R2:", summary.r2)
print("Train RMSE:", summary.rootMeanSquaredError)
print("Train MAE:", summary.meanAbsoluteError)

Train R2: 0.986718024635141
Train RMSE: 25009.13070168998
Train MAE: 19954.007938409617
